<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/AdvancedAgent/ipynb/2.1_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-openai langchain-mcp-adapters \
    tavily-python mcp

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
import dotenv

os.makedirs("resources", exist_ok=True)
_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(),
        asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

## Local MCP server

In [3]:
%%writefile resources/2.1_mcp_server.py
import dotenv
from mcp.server import fastmcp
import tavily
import typing
import requests

_ = dotenv.load_dotenv(dotenv_path="../env", override=True)

mcp = fastmcp.FastMCP("mcp_server")

tavily_client = tavily.TavilyClient()
# Tool for searching the web
@mcp.tool()
def search_web(query: str) -> typing.Dict[str, typing.Any]:
    """
    Search the web for information
    """
    results = tavily_client.search(query)
    return results

# Resources - provide access to langchain-ai repo files
@mcp.resource("github://langchain-ai/langchain-mcp-adapters/main/README.md")
def github_file():
    """
    Resource for accessing langchain-ai/langchain-mcp-adapters/README.md file
    """
    url = "https://raw.githubusercontent.com/langchain-ai/langchain-mcp-adapters/main/README.md"
    try:
        response = requests.get(url)
        return response.text
    except Exception as e:
        return f"Error: {str(e)}"

# Prompt template
@mcp.prompt()
def prompt():
    """
    Analyze data from a langchain-ai repo file with comprehensive insights
    """
    return """
    You are a helpful assistant that answer user questions about Langchain,
    LangGraph and LangSmith.

    You can use the following tools/resources to answer user questions:
    - search_web: Search the web for information
    - github_file: Access the langchain-ai repo files

    If the user asks a question that is not related to LangChain, LangGraph or
    LangSmith, you should say "I'm sorry, I can only answer questions about
    LangChain, LangGraph and LangSmith."

    You may try multiple tool and resource calls to answer the user's question.

    You may also ask clarifying questions to the user to better understand
    their question.
    """

if __name__ == "__main__":
    mcp.run(transport="sse")

Overwriting resources/2.1_mcp_server.py


In [4]:
!nohup python resources/2.1_mcp_server.py &

nohup: appending output to 'nohup.out'


In [5]:
from langchain_mcp_adapters import client

client = client.MultiServerMCPClient({
    "local_server": {
        "transport": "sse",
        "url": "http://127.0.0.1:8000/sse"
    }
})

# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [6]:
import os
import time
from langchain import chat_models, agents, messages

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

agent = agents.create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt
)

config = {"configurable": {"thread_id": "1"}}
start_time = time.time()
response = await agent.ainvoke(
    {"messages": [messages.HumanMessage(content="""
        Tell me about the langchain-mcp-adapters library
    """)]},
    config=config
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 27.38s
================================ Human Message =================================


        Tell me about the langchain-mcp-adapters library
    
================================== Ai Message ==================================
Tool Calls:
  search_web (call_39u3zs1e)
 Call ID: call_39u3zs1e
  Args:
    query: langchain-mcp-adapters
================================= Tool Message =================================
Name: search_web

[{'type': 'text', 'text': '{\n  "query": "langchain-mcp-adapters",\n  "follow_up_questions": null,\n  "answer": null,\n  "images": [],\n  "results": [\n    {\n      "url": "https://changelog.langchain.com/announcements/mcp-adapters-for-langchain-and-langgraph",\n      "title": "MCP Adapters for LangChain and LangGraph - Changelog",\n      "content": "# LangChain Changelog. Sign up for our newsletter to stay up to date. # MCP Adapters for LangChain and LangGraph. The **LangChain MCP Adapters** is a package that makes it easy to use **Anthropic 

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)